# Part 1: Building a Local Large Language Model Application

This guide provides instructions for setting up and running a local Large Language Model (LLM) application using Groq and LlamaIndex.

> This notebook is a guide to setting up this project on your local machine. **Do not run the code cells here, you will get an error.**

---
## 1.&nbsp; Project Setup and Library Installation

First, we'll get your project folder and all the necessary Python libraries set up. It's good practice to create a dedicated, self-contained environment for each project. This ensures that all the required libraries are the correct versions and won't conflict with other projects you might be working on.

1. Let's begin by creating the main folder for our application. Find a suitable place on your hard drive, like your 'Documents' or a dedicated 'Projects' folder, and create a new directory called **rag_project**. Once you've created it, open this new folder in VSCode.

2. With the project folder open, the next step is to create the Conda environment. We've simplified this process by providing an `environment.yml` file. This file acts as a blueprint, listing all the libraries and dependencies the project needs to run correctly.

    You'll need to download this file and place it directly inside your **Rag Project** folder.

3. Now that the environment file is in your project folder, you can create the environment. Open the terminal within VSCode (you can do this by going to the 'Terminal' menu and selecting 'New Terminal'). In the terminal, run the following command. This tells Conda to read the blueprint file and install everything listed inside it.

In [ ]:
conda env create -f environment.yml

This process might take a few minutes as it downloads and installs all the packages. Once it's complete, you'll need to activate the new environment so that your project can use it. Run this command in the terminal:

In [ ]:
conda activate rag-project-env

Your terminal prompt should now change to show `(rag_project_env)` at the beginning. This confirms that the environment is active and you're ready to start building the application.

> It's a good idea to set this as your Python Interpreter in VSCode, too. While your terminal is set up, VSCode's features like debugging and code-completion run separately, so they won't find your newly installed packages unless you tell VSCode which environment to use. Just press `cmd + shift + p`, select `Python: Select Interpreter`, and choose your new `rag-project-env` from the list.

---
## 2.&nbsp; API Key Configuration

To get our LlamaIndex project running, we need to connect it to an actual Large Language Model. We'll be using one of Groq's high-performance models, and to do that, our application needs an API key to authenticate its requests. We'll put this key in a `.env` file, which is a secure way to store sensitive secrets (like API keys or passwords) without writing them directly into our code where they could be accidentally shared.

### 2.1. Create the .env file in VS Code

Your `.env` file is just a plain text file you'll use to store secrets, like API keys, so they aren't written directly into your shared code.

1.  In VS Code, open the File Explorer on the left side (the icon showing two documents).
2.  Make sure you're in the main (root) folder for your project, not inside another sub-folder.
3.  Click the 'New File' icon.
4.  When prompted for a name, type exactly `.env` (starting with the dot is important) and press Enter.

> If you're using git, it's also a good practice to add `.env` to your `.gitignore` file if you're using Git, so you don't accidentally upload your secret keys.
>
> If you're not using git, make sure you don't upload your `.env` file to GitHub at the end of this project.

### 2.2. Get your Groq API Key

Next, you'll need to get the key from Groq.

1.  Go to the URL: `https://console.groq.com/keys`
2.  You'll need to sign in (you can usually use your Google or GitHub account).
3.  Once you're in, look for the button to 'Create API Key'.
4.  Give your key a name if requested.
5.  Groq will generate a string of characters. This is your key. Copy this key immediately - for security, they often won't show it to you again.

### 2.3. Store the Key in your .env file

Now, let's put the key you just copied into the file you made.

1.  Go back to VS Code and open your new `.env` file.
2.  Type a variable name for your key, like `GROQ_API_KEY`, followed immediately by an equals sign (`=`).
3.  Paste your API key right after the equals sign.

    It's important that you don't use any spaces around the `=` or put any quotes around the key. It should look just like this (but with your real key):

```
GROQ_API_KEY=gsk_AbCdEfGhIjKlMnOpQrStUvWxYz1234567890
```

4.  Save the `.env` file and you're all set! Your Python code will now be able to load this key from the environment.

---
## 3.&nbsp; Project Structure and Configuration

We'll now create the necessary files and directories. This includes a configuration file to manage model parameters and Python package initialisers.

### 3.1. Initialise the Python Package

In the VSCode file explorer, create an directory called `src`. Then create an empty file named `__init__.py` inside the `src` directory.

This file is special. Its presence turns the `src` directory into a Python 'package'. This is what allows you to import code from other files within this folder. Even though newer Python versions are more flexible, adding this empty file is still the standard way to do things. It makes your project structure clear and ensures everything works smoothly with other development tools.

### 3.2. Create the Configuration File

1. Inside the `src` directory, create a new file named `config.py`.
2. In the VSCode editor, add the following code to `src/config.py`. This file centralises all model parameters for easy management.

In [ ]:
# --- LLM Model Configuration ---
LLM_MODEL: str = "llama-3.3-70b-versatile"
LLM_MAX_NEW_TOKENS: int = 768
LLM_TEMPERATURE: float = 0.01
LLM_TOP_P: float = 0.95
LLM_REPETITION_PENALTY: float = 1.03
LLM_QUESTION: str = "What is the capital of France?"


4. Save the file.

The `config.py` file serves as a centralised hub for all the settings and parameters that control your application's behaviour. Its primary purpose is to separate configuration from code.

Instead of hardcoding values like model names, file paths, or numerical settings directly into your functions, you define them as constants in `config.py`.

This means you can easily change how the application works (e.g., switch to a different model) by editing just one file, without having to hunt through the logic of all your code. All the important variables that define your RAG pipeline's setup are in one predictable location, making the project much easier to manage and maintain.

---
## 4.&nbsp; Create the Engine and its Components

Now we'll build the core of our application. Instead of putting all our logic in one file, we will apply a key software design principle called Separation of Concerns. This will make our project cleaner, more scalable, and easier to understand.

### 4.1 Separation of Concerns

Separation of Concerns means we should break down a complex application into distinct sections, where each section is responsible for one specific job or "concern".

In our project, we'll organise our code into two main files:

1. `model_loader.py`

      * Concern: Initialisation.
      * Responsibility: Its only job is to create the fundamental, low-level tools our application needs. It knows how to connect to services by fetching API keys and which specific models to load. Think of it as our project's parts store; its functions produce the raw "parts".

2.  `engine.py`

      * Concern: Orchestration.
      * Responsibility: Its job is to take the "parts" and assemble them into a final, working application. It's the assembly line where the individual components are put together to create the finished product: our chat engine.

By separating our code this way, we create a system that is more modular and easier to manage.


### 4.2 `model_loader.py`

Let's create the file that will be responsible for initialising our components.

1.  In the VSCode file explorer, create a new file named `model_loader.py` inside the `src` directory.
2.  In the VSCode editor, add the following code to `src/model_loader.py`.

In [ ]:
import os
from dotenv import load_dotenv

from llama_index.llms.groq import Groq

from src.config import (
    LLM_MODEL,
    # LLM_MAX_NEW_TOKENS,
    # LLM_TEMPERATURE,
    # LLM_TOP_P,
    # LLM_REPETITION_PENALTY
)


# Load environment variables from the .env file
load_dotenv()


def initialise_llm() -> Groq:
    """Initialises the Groq LLM with core parameters from config."""

    api_key: str | None = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError(
            "GROQ_API_KEY not found. Make sure it's set in your .env file."
        )

    return Groq(
        api_key=api_key,
        model=LLM_MODEL,
        # The following parameters are optional
        # and will default to the model's defaults if not set
        # max_tokens=LLM_MAX_NEW_TOKENS,
        # temperature=LLM_TEMPERATURE,
        # top_p=LLM_TOP_P,
    )


3.  Save the file.

This script's only job, at the moment, is to create a configured `Groq` LLM object. It reads the API key from the environment and the model name from our `config.py` file, creating a reusable "component" for our application.

### 4.4 `engine.py`

Now we'll create the engine file that uses the component we just built.

1.  In the VSCode file explorer, create a new file named `engine.py` inside the `src` directory.
2.  In the VSCode editor, add the following code to `src/engine.py`.

In [ ]:
from llama_index.core.llms import CompletionResponse, LLM

from src.config import LLM_QUESTION
from src.model_loader import initialise_llm


def main_chat_loop() -> None:
    """Main chat loop to ask a question to the LLM and print the answer."""

    llm: LLM = initialise_llm()

    answer: CompletionResponse = llm.complete(LLM_QUESTION)

    print(f"Question: {LLM_QUESTION}")
    print("---")
    print(f"Answer: {answer}")


3.  Save the file.

> Notice how clean this file is. It doesn't need to know about API keys or model names. Its only job is to orchestrate the application flow: it imports the `initialise_llm` function from our `model_loader` module, calls it to get the LLM, and then uses the LLM to run the chat loop.

You've now successfully created the core engine logic with a clean separation of concerns. In the next step, we will create the main `main.py` file to run our application.

---
## 5.&nbsp; Create the Main Application Entry Point

A `main.py` file serves as the primary entry point for the application. This standard practice makes the project's starting point clear and easy to execute.

1. In the VSCode file explorer, create a new file named `main.py` in the project's root folder.
2. In the VSCode editor, add the following code to `main.py`.
3. Save the file.

In [ ]:
from src.engine import main_chat_loop


def main() -> None:
    print("--- 🤖 Main Application Starting ---")
    # Start the main chat session
    main_chat_loop()


if __name__ == "__main__":
    main()


> `if __name__ == "__main__"` is a standard and important convention in Python. It checks if the script is being executed directly by the user (e.g., `python main.py`).
>
> If it is, the code inside the block (in this case, calling our main() function) will run. This setup prevents the code from running automatically if this file is ever imported as a module into another script, making your code more reusable and predictable.

---
## 6.&nbsp; Run the Application

With all the components in place, run the application from the VSCode terminal.

In [ ]:
python main.py

Congratulations, you've successfully built and run your LLM.

---
## 7.&nbsp; Challenges and Further Exploration 😀

Congratulations on building a functioning LLM application! The best way to learn is by experimenting. This section provides some challenges to help you explore the capabilities of your new application and understand the impact of different configurations.

Remember to **only change one parameter at a time**. This will help you isolate the effect of each change and develop a better intuition for how they influence the model's output.

### 1. Experiment with Different Models

Groq offers access to a variety of high-performance models. Experiment with how to change a model, and with how different models respond to the same questions.

How to Change the Model:

1.  Open the `src/config.py` file in your VSCode editor.
2.  Locate the `LLM_MODEL` variable.
3.  Replace the current model ID with a new one from the list below.
4.  Make sure you save the changes to `src/config.py` or you won't change the model.

You can find a complete and up-to-date list of available models on the [Groq Models Rate Limits Page](https://console.groq.com/docs/rate-limits).

> For your project, you'll want to find a model that has a good balance between its daily and minute-by-minute request limits.
>
> We suggest looking for:
>
> * Requests per minute (RPM): 20 or more
> * Requests per day (RPD): 250 or more
>
> You'll find there's a trade-off here. Since we're using the free tier, the more advanced models often have tighter limits because they require more computing power. For example, a top-tier model might only allow two requests per minute, which makes frequent testing difficult. You'll need to choose a model that's smart enough for what you want to do, but also has high enough limits that you aren't blocked when you're constantly running your code.


**Challenge:** Try the same question (`LLM_QUESTION`) with at least three different models - maybe something trickier than just the capital of France will help you spot differences. Take notes on the differences in speed, detail, and phrasing in their answers.

### 2. Adjust Model Parameters

The parameters in your `src/config.py` file give you fine-grained control over the model's response generation.

* `LLM_MAX_NEW_TOKENS`
    * What it does: This sets the absolute maximum number of tokens (which are like pieces of words) that the model is allowed to generate for its answer.
    * Experiment: Try setting this to a very low value (e.g., `50`) and see how the model cuts its response short. Then, try a much larger value (e.g., `2000`) and ask a complex question to see if it produces a longer, more detailed response.

* `LLM_TEMPERATURE`
    * What it does: This controls the randomness of the output. A higher temperature (e.g., `0.8`) makes the output more creative and unpredictable, as the model is more likely to choose less probable words. A lower temperature (e.g., `0.2`) makes the responses more deterministic and focused.
    * Experiment: Set the temperature to `0.9` and run the same prompt a few times. You should see more variation in the answers. Now, set it to `0.0` and run it again; the answers should be nearly identical each time.

* `LLM_TOP_P`
    * What it does: This is an alternative method to control randomness, often used instead of temperature. It works by selecting from the most probable words whose cumulative probability exceeds a certain threshold (`top_p`). A higher value (e.g., `0.95`) gives the model more words to choose from, leading to more creativity. A lower value (e.g., `0.5`) restricts the choices to a smaller, more probable set of words.
    * Experiment: Try setting `LLM_TEMPERATURE` to `1.0` and `LLM_TOP_P` to `0.1`. The model will be highly creative but will be forced to pick from a very narrow set of top words, which can produce interesting results.

* `LLM_REPETITION_PENALTY`
    * What it does: This parameter discourages the model from repeating the same words or phrases. A value greater than `1.0` (e.g., `1.2`) will apply a penalty to words that have already appeared, making them less likely to be chosen again.
    * Experiment: Ask the model a question that might lead to repetitive text. First, run it with the penalty at `1.0` (no penalty). Then, increase it to `1.3` and observe if the output becomes less repetitive and more diverse.

By methodically adjusting these settings, you will gain a practical understanding of how to tailor an LLM's behaviour to fit your specific needs. Happy experimenting!
